In [4]:
from pathlib import Path
import pandas as pd

# Run from the folder containing this notebook and the two supplied CSV files.
files = {2025: "2025.csv", 2026: "2026PartYear.csv"}
frames = {}
for year, filename in files.items():
    df = pd.read_csv(Path(filename))
    us = df.loc[df["Geography"].eq("Total U.S.") & df["Period"].between(1, 32)].copy()
    assert us.groupby("Type")["Period"].nunique().to_dict() == {"Conventional": 32, "Organic": 32}
    frames[year] = us

def summarize(df):
    volume = df["Total Bulk and Bags"].sum()
    return {"volume": volume,
            "ASP": (df["ASP Current Year"] * df["Total Bulk and Bags"]).sum() / volume}

rows = []
for year, df in frames.items():
    all_us = summarize(df)
    for group, subset in [("Total U.S.", df),
                          ("Conventional", df.loc[df["Type"].eq("Conventional")]),
                          ("Organic", df.loc[df["Type"].eq("Organic")])]:
        m = summarize(subset)
        rows.append({"Year": year, "Group": group, "Volume": m["volume"], "Weighted ASP": m["ASP"],
                     "Bagged": subset["TotalBagged"].sum(),
                     **{f"PLU {plu}": subset[plu].sum() for plu in ["4046", "4225", "4770"]}})
summary = pd.DataFrame(rows).set_index(["Year", "Group"])
for year in files:
    total = summary.loc[(year, "Total U.S."), "Volume"]
    summary.loc[(year, "Total U.S."), "Bagged share %"] = 100 * summary.loc[(year, "Total U.S."), "Bagged"] / total
    summary.loc[(year, "Organic"), "Organic share %"] = 100 * summary.loc[(year, "Organic"), "Volume"] / total

def value(year, group, metric):
    return summary.loc[(year, group), metric]

def change(group, metric):
    return 100 * (value(2026, group, metric) / value(2025, group, metric) - 1)

# Memo claims: displayed numbers are rounded; the tolerance reflects their stated precision.
claims = [
    ("Total volume 2025 (billions)", value(2025, "Total U.S.", "Volume") / 1e9, 1.871, .0005),
    ("Total volume 2026 (billions)", value(2026, "Total U.S.", "Volume") / 1e9, 2.126, .0005),
    ("Total volume change %", change("Total U.S.", "Volume"), 13.6, .05),
    ("Weighted ASP 2025 ($)", value(2025, "Total U.S.", "Weighted ASP"), 1.25, .005),
    ("Weighted ASP 2026 ($)", value(2026, "Total U.S.", "Weighted ASP"), 1.01, .005),
    ("Weighted ASP change %", change("Total U.S.", "Weighted ASP"), -19.1, .05),
    ("Conventional volume 2025 (billions)", value(2025, "Conventional", "Volume") / 1e9, 1.765, .0005),
    ("Conventional volume 2026 (billions)", value(2026, "Conventional", "Volume") / 1e9, 2.021, .0005),
    ("Conventional volume change %", change("Conventional", "Volume"), 14.5, .05),
    ("Conventional ASP 2025 ($)", value(2025, "Conventional", "Weighted ASP"), 1.22, .005),
    ("Conventional ASP 2026 ($)", value(2026, "Conventional", "Weighted ASP"), .98, .005),
    ("Conventional ASP change %", change("Conventional", "Weighted ASP"), -19.4, .05),
    ("Bagged 2025 (millions)", value(2025, "Total U.S.", "Bagged") / 1e6, 782.3, .05),
    ("Bagged 2026 (millions)", value(2026, "Total U.S.", "Bagged") / 1e6, 919.8, .05),
    ("Bagged change %", change("Total U.S.", "Bagged"), 17.6, .05),
    ("Bagged share 2025 %", value(2025, "Total U.S.", "Bagged share %"), 41.8, .05),
    ("Bagged share 2026 %", value(2026, "Total U.S.", "Bagged share %"), 43.3, .05),
    ("Organic volume 2025 (millions)", value(2025, "Organic", "Volume") / 1e6, 105.9, .05),
    ("Organic volume 2026 (millions)", value(2026, "Organic", "Volume") / 1e6, 105.1, .05),
    ("Organic volume change %", change("Organic", "Volume"), -.7, .05),
    ("Organic ASP 2025 ($)", value(2025, "Organic", "Weighted ASP"), 1.74, .005),
    ("Organic ASP 2026 ($)", value(2026, "Organic", "Weighted ASP"), 1.56, .005),
    ("Organic ASP change %", change("Organic", "Weighted ASP"), -10.3, .05),
    ("Organic share 2025 %", value(2025, "Organic", "Organic share %"), 5.7, .05),
    ("Organic share 2026 %", value(2026, "Organic", "Organic share %"), 4.9, .05),
]
for plu, old, new, pct in [("4046", 867.2, 889.1, 2.5),
                            ("4225", 197.3, 267.5, 35.6),
                            ("4770", 23.9, 49.5, None)]:
    metric = f"PLU {plu}"
    claims += [(f"{metric} 2025 (millions)", value(2025, "Total U.S.", metric) / 1e6, old, .05),
               (f"{metric} 2026 (millions)", value(2026, "Total U.S.", metric) / 1e6, new, .05)]
    if pct is not None:
        claims.append((f"{metric} change %", change("Total U.S.", metric), pct, .05))

checks = pd.DataFrame(claims, columns=["Memo claim", "Calculated", "Memo value", "Tolerance"])
checks["Result"] = checks.apply(lambda r: "✓" if abs(r["Calculated"] - r["Memo value"]) < r["Tolerance"] else "REVIEW", axis=1)
print(checks.to_string(index=False, formatters={"Calculated": "{:,.4f}".format}))
print(f"\nVerified: {(checks['Result'] == '✓').sum()}/{len(checks)} claims")
print(f"PLU 4770 change: {change('Total U.S.', 'PLU 4770'):.1f}% (memo says more than doubled)")


                         Memo claim Calculated  Memo value  Tolerance Result
       Total volume 2025 (billions)     1.8707       1.871     0.0005      ✓
       Total volume 2026 (billions)     2.1258       2.126     0.0005      ✓
              Total volume change %    13.6367      13.600     0.0500      ✓
              Weighted ASP 2025 ($)     1.2452       1.250     0.0050      ✓
              Weighted ASP 2026 ($)     1.0079       1.010     0.0050      ✓
              Weighted ASP change %   -19.0516     -19.100     0.0500      ✓
Conventional volume 2025 (billions)     1.7648       1.765     0.0005      ✓
Conventional volume 2026 (billions)     2.0207       2.021     0.0005      ✓
       Conventional volume change %    14.4968      14.500     0.0500      ✓
          Conventional ASP 2025 ($)     1.2154       1.220     0.0050      ✓
          Conventional ASP 2026 ($)     0.9791       0.980     0.0050      ✓
          Conventional ASP change %   -19.4422     -19.400     0.0500      ✓